In [1]:
import os
import requests

from dotenv import load_dotenv

from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_ollama import ChatOllama


C:\Users\Hp\AppData\Local\Temp\ipykernel_21980\1708253669.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


In [2]:
load_dotenv()

SERPER_API_KEY = os.getenv("SERPER_API_KEY")


In [3]:
embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [4]:
loader = TextLoader("usa.txt")

documents = loader.load()


In [5]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

chunks = splitter.split_documents(documents)


In [6]:

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding,
    persist_directory="chroma_db"
)


In [7]:

# Search Chroma Database

def vector_search(query):

    results = vectorstore.similarity_search_with_score(
        query,
        k=2
    )

    docs = []

    for doc, score in results:

        if score < 1.0:
            docs.append(doc.page_content)

    return "\n".join(docs)


In [8]:

# Web Search using Serper

def web_search(query):

    url = "https://google.serper.dev/search"

    headers = {
        "X-API-KEY": SERPER_API_KEY,
        "Content-Type": "application/json"
    }

    data = {
        "q": query
    }

    response = requests.post(
        url,
        json=data,
        headers=headers
    )

    results = response.json()

    snippets = []

    for item in results.get("organic", []):

        snippet = item.get("snippet", "")

        if snippet:
            snippets.append(snippet)

    return "\n".join(snippets[:5])


In [9]:

#  Retriever Agent

def retriever_agent(query):

    print("\nSearching Chroma DB...")

    context = vector_search(query)

    if context:

        print("Found answer in Chroma DB")

        return context

    print("Not found in Chroma DB")

    print("Searching the Web...")

    return web_search(query)



In [10]:
llm = ChatOllama(
    model="llama3.2"
)


In [11]:
def generate_answer(query, context):

    prompt = f"""
Answer the question using the context below.

Context:
{context}

Question:
{query}

Give a clear and concise answer.
"""

    response = llm.invoke(prompt)

    return response.content


In [13]:

query = input("\nAsk a question: ")

context = retriever_agent(query)

answer = generate_answer(
    query,
    context
)

print("\nFinal Answer:")
print(answer)


Ask a question:  GDP of USA in 2026



Searching Chroma DB...
Found answer in Chroma DB

Final Answer:
Unfortunately, the provided context does not include information about the GDP of the United States in 2026. The context only provides data up to 2024. Therefore, it is not possible to provide a clear and concise answer about the GDP of the United States in 2026 based on the given information.
